# Phase 1 — Dataset Acquisition & Exploratory Data Analysis (CIC-IDS2017)

**Status:** ✅ Completed (pipeline v1.0)  
**Authoritative Script:** `ml_service/src/eda.py`  
**Source Artifact:** `artifacts/eda_summary.json` & `artifacts/eda_summary.md`  

---

## Executive Summary & Research Context
This notebook provides an interactive audit of the **CIC-IDS2017** benchmark dataset. The dataset comprises **8 daily capture CSVs** totaling **2,830,743 network flows** with **79 flow attributes**.

### Core Exploration Objectives:
1. **Class Distribution & Imbalance:** Audit benign baseline vs. 14 distinct attack subtypes.
2. **Data Hygiene & Defects:** Measure exact occurrences of leading column whitespace, `NaN` values, and `+Infinity` byte/packet rates.
3. **Cryptographic Duplicate Analysis:** Evaluate 128-bit MD5 duplicate flow collisions (within-file vs cross-file).

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is importable
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from config.paths import Paths

with open(Paths.ARTIFACTS_DIR / 'eda_summary.json', 'r', encoding='utf-8') as f:
    eda = json.load(f)

print(f"EDA Generated:     {eda['metadata']['timestamp']}")
print(f"Files Analyzed:    {eda['metadata']['files_analyzed_count']}")
print(f"Total Raw Flows:   {eda['global_row_and_duplicate_metrics']['total_rows_across_all_files']:,}")
print(f"Global Unique:     {eda['global_row_and_duplicate_metrics']['global_unique_rows']:,}")
print(f"Global Duplicates: {eda['global_row_and_duplicate_metrics']['global_duplicate_rows']:,} ({eda['global_row_and_duplicate_metrics']['global_duplicate_percentage']}%)")

## 1. Daily Capture Breakdown & Security Scenario Mapping
Each CSV represents a specific operational day and designated attack injection scenario:

In [ ]:
file_records = []
for f_rep in eda['per_file_reports']:
    file_records.append({
        'File Name': f_rep['file_name'],
        'Total Rows': f_rep['row_count'],
        'Unique Rows': f_rep['unique_row_count'],
        'Duplicate %': f"{f_rep['duplicate_percentage']:.2f}%",
        'NaN Count': f_rep['nan_count'],
        '+Inf Count': f_rep['pos_inf_count'],
        'File Size (MB)': f"{f_rep['file_size_bytes'] / (1024**2):.1f}"
    })

df_files = pd.DataFrame(file_records)
df_files.style.format({'Total Rows': '{:,}', 'Unique Rows': '{:,}'})

## 2. Attack Category Distribution & Extreme Class Imbalance
Visualizing the class distribution across all categories on a **logarithmic scale** reveals the extreme minority attack classes (Heartbleed $N=11$, Infiltration $N=36$, SQLi $N=21$) compared to high-volume DoS ($N>230\text{k}$) and Benign ($N>2.27\text{M}$).

In [ ]:
class_dist = eda['global_class_distribution']
df_classes = pd.DataFrame(list(class_dist.items()), columns=['Category', 'Count']).sort_values('Count', ascending=False)

plt.figure(figsize=(12, 6))
sns.set_theme(style='whitegrid')
barplot = sns.barplot(data=df_classes, y='Category', x='Count', hue='Category', palette='mako', legend=False)
plt.xscale('log')
plt.title('CIC-IDS2017 Global Class Distribution (Log Scale)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Flow Count (Logarithmic Scale)', fontsize=12)
plt.ylabel('Threat / Attack Category', fontsize=12)

for p in barplot.patches:
    width = p.get_width()
    if width > 0:
        barplot.annotate(f'{int(width):,}',
                         (width, p.get_y() + p.get_height() / 2.),
                         ha='left', va='center',
                         xytext=(5, 0), textcoords='offset points',
                         fontsize=9, fontweight='semibold')

plt.tight_layout()
plt.show()

## 3. Data Hygiene: Missing Values, Infinite Rates, and Header Whitespace
- **Whitespace Audit:** 65 of 79 columns in the raw dataset possess leading whitespace (e.g. `' Destination Port'`, `' Label'`).
- **Rate Calculation Anomalies:** `Flow Bytes/s` contains 1,358 NaNs and 1,509 `+inf` values; `Flow Packets/s` contains 2,867 `+inf` values, caused by zero duration flows.

In [ ]:
hygiene_data = {
    'Defect Type': ['Leading Column Whitespace', 'NaN Values (Flow Bytes/s)', '+Infinity Rates (Packets/s)', '+Infinity Rates (Bytes/s)', 'Cross-File Duplicates'],
    'Affected Scope': ['65 / 79 Columns', '1,358 Flows', '2,867 Flows', '1,509 Flows', '52,902 Overlapping Flows'],
    'Root Cause in Capture': ['CSV generator formatting quirk', '0 Duration flow byte division', '0 Duration flow packet division', '0 Duration flow byte division', 'Inter-day persistent session capture'],
    'Pipeline Remedy (Phase 2)': ['strip() all headers', 'Drop corrupt rows', 'Drop corrupt rows', 'Drop corrupt rows', '128-bit hash global deduplication']
}
pd.DataFrame(hygiene_data)